# ALMA-C11 + CEERS analogues — SED comparison, attenuation, torus inclination

Analysis companion of `analogues_specphot_almac11_rt.ipynb`. Reads ONLY through
`almac11_specphot/almac11_source_map.fits` (built by the staging notebook's Part 6 census), so
reused CEERS trees and new `sed_almac11` trees are transparent.

**Sightline convention.** The dust arms (`dust_on`/`dust_off`) carry 4 sightlines
(`THETA=[0,45,90,135]`), the Nenkova arms a single one (`THETA=[0]`, `findx=0`); the reused
CEERS `nenkova_i90` tree carries 4. Whenever arms are differenced per galaxy
(attenuation, torus residuals) the dust arms are read at `findx=0` and multi-sightline Nenkova
SEDs are averaged; for stacks against observations the sightline mean is used (<0.5% scatter
for these dust-poor hosts either way).

**Observed side.** The CSV carries the CIGALE best-model fluxes (`best.<band>`, mJy) for
19 bands (MegaCam/Suprime u->i, VISTA JHKs, IRAC 1-4, MIPS 24, PACS 100, SPIRE 250, ALMA band 6)
— plotted at band pivot wavelengths. Observed photometry with errors is NOT in the CSV; the
`OBS_PHOT_HOOK` below is the place to point at a staged CIGALE `observations.fits` when copied
to the cluster. AGN properties (L_AGN for the torus-rescaling panel) live in
`ALMAC11_target+control_jwst_AGN_prop.fits` — local only as of 2026-08-04; copy to
`obs_data/almac11/` to enable that panel.

**Parts**: 1 observed side · 2 mock stacks per target · 3 attenuation A_lambda / A_V vs CIGALE
Av_ISM · 4 torus MIR vs inclination (i=90/60/30) · 5 dust masses (sanitized CSV column) ·
6 AGN necessity + obscuration vs observed bands · 7 dust ratios vs attenuation ·
8 host-sightline A_V spread · 9 **consistent A_V** — CIGALE fits of the mock dust_on
photometry in the observed 19 bands (SLURM array; closes the Part 3 estimator mismatch).

**Science questions** (the point of the exercise — analogues were selected in the age–mass
windows where SIMBA predicts dusty sources, dust-to-molecular > 10⁻³ vs controls < 10⁻⁴):
1. Does SIMBA need an **AGN** to reproduce the observed SEDs? → Part 6
2. If so, does the AGN need to be **obscured** (torus i=90 vs 60 vs 30)? → Part 6
3. When the **dust ratios** (dust-to-stellar, dust-to-gas) match, does the **attenuation**
   (Av_ISM) follow? → Part 7, like-for-like in Part 9
4. Does any of this require a particular **inclination** — host sightline (Part 8) or torus
   inclination (Part 6)?


In [ ]:
# ── Part 0 · config + resolvers ──────────────────────────────────────────────
import os, re, glob
from collections import defaultdict

import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from hyperion.model import ModelOutput

HOME       = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR     = os.path.join(HOME, 'output', 'cis100', 'almac11_specphot')
FIGDIR     = os.path.join(OUTDIR, 'figures')
ALMAC11_CSV = os.path.join(HOME, 'ALMAC11_sed_ism_modeling_results.csv')
OBS_PHOT_HOOK = None   # -> staged CIGALE observations.fits for the ALMA-C11 sample (optional)
AGN_PROP_HOOK = os.path.join(HOME, 'obs_data', 'almac11',
                             'ALMAC11_target+control_jwst_AGN_prop.fits')  # optional
os.makedirs(FIGDIR, exist_ok=True)

ARMS = ('dust_on', 'dust_off', 'nenkova_i90', 'nenkova_i60', 'nenkova_i30')
COL = {'no_AGN': 'tab:blue', 'AGN': 'tab:orange', 'obs': 'k',
       'nenkova_i90': '#b30000', 'nenkova_i60': '#e34a33', 'nenkova_i30': '#fc8d59'}

# ── targets: CSV rows + the two CEERS entries ────────────────────────────────
C = Table.read(ALMAC11_CSV, format='csv')
TARGETS = {}
for r in C:
    tid = str(r['ID']).strip()
    TARGETS[tid] = dict(z=float(r['z']), logm=float(r['log(Mstar/Msol)']),
                        age_gyr=10.0 ** float(r['Age(log/yr)']) / 1e9,
                        av_ism=float(r['bayes.attenuation.Av_ISM']),
                        av_ism_err=float(r['bayes.attenuation.Av_ISM_err']),
                        sample='almac11', row=dict(zip(r.colnames, r)))
TARGETS['719']  = dict(z=1.463, logm=10.29, age_gyr=2.65, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', row=None)
TARGETS['2962'] = dict(z=1.266, logm=10.68, age_gyr=3.30, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', row=None)
_tid_safe = lambda tid: re.sub(r'[^A-Za-z0-9_-]', '', tid)

# ── selection tables + source map ────────────────────────────────────────────
SELECTED = {tid: Table.read(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))
            for tid in TARGETS
            if os.path.exists(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))}
with fits.open(os.path.join(OUTDIR, 'rt_union.fits')) as _h:
    UNION = Table(_h['UNION'].data)
    MEMBERS = Table(_h['MEMBERS'].data)
SMAP = Table.read(os.path.join(OUTDIR, 'almac11_source_map.fits'))
for _t in (UNION, MEMBERS, SMAP):          # FITS strings load as bytes; decode once
    _t.convert_bytestring_to_unicode()

_smap = {}
for r in SMAP:
    if bool(r['COMPLETE']):
        _smap[(str(r['ARM']), int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))] = str(r['RTOUT_PATH'])

def rtout_path(arm, snap, gid):
    return _smap.get((arm, int(snap), int(gid)))   # None if RT not complete

_as_str = lambda col: np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x)
                                for x in col])

# ── SED reader (largest aperture; sightline handling per convention above) ───
_SED_CACHE = {}
def read_sed(arm, snap, gid, sightline='mean'):
    # -> (wav_rest_um, nuLnu_erg_s) or None if the RT is not complete.
    key = (arm, int(snap), int(gid), sightline)
    if key not in _SED_CACHE:
        p = rtout_path(arm, snap, gid)
        if p is None:
            return None
        m = ModelOutput(p)
        sed = m.get_sed(inclination='all', aperture=-1)
        wav = np.asarray(sed.wav, float)
        val = np.atleast_2d(np.asarray(sed.val, float))
        val = val[0] if (sightline == 'findx0' or val.shape[0] == 1) else val.mean(axis=0)
        s = np.argsort(wav)
        _SED_CACHE[key] = (wav[s], val[s])
    return _SED_CACHE[key]

GRID = np.geomspace(0.08, 1500.0, 400)   # rest-frame micron

def on_grid(wl, nuLnu):
    with np.errstate(invalid='ignore', divide='ignore'):
        return 10 ** np.interp(np.log10(GRID), np.log10(wl), np.log10(nuLnu),
                               left=np.nan, right=np.nan)

def attenuation_curve(snap, gid):
    # A_lambda on GRID from the dust_on/dust_off findx0 pair (None if RT missing)
    on = read_sed('dust_on', snap, gid, sightline='findx0')
    off = read_sed('dust_off', snap, gid, sightline='findx0')
    if on is None or off is None:
        return None
    with np.errstate(invalid='ignore', divide='ignore'):
        return -2.5 * np.log10(on_grid(*on) / on_grid(*off))

_jV = int(np.argmin(np.abs(GRID - 0.55)))    # V-band index on GRID

def obs_fnu_factor(z):
    # (lambda_obs [um], factor nuLnu[erg/s] -> Fnu [mJy]) at redshift z
    wav_obs = GRID * (1.0 + z)
    dl = Planck13.luminosity_distance(z).to(u.cm).value
    nu_obs = (const.c / (wav_obs * u.micron)).to(u.Hz).value
    return wav_obs, 1.0 / (4.0 * np.pi * dl**2 * nu_obs) / 1e-26

print(f'{len(TARGETS)} targets | {len(UNION)} union galaxies | '
      f'{len(SMAP)} source-map rows ({int(np.asarray(SMAP["COMPLETE"]).sum())} complete)')
for arm in ARMS:
    m_ = np.asarray(SMAP['ARM']) == arm
    #print(f'  {arm:12s} {int(np.asarray(SMAP["COMPLETE"])[m_].sum()):4d} / {int(m_.sum()):4d} complete')


## Part 1 · observed side — CIGALE best-model fluxes at band pivots

`best.<band>` are model fluxes in mJy (no errors). Pivot wavelengths below are observed-frame
microns; replace with the exact CIGALE filter pivots if a filters directory is staged.


In [ ]:
# ── Part 1 · observed photometry (CIGALE best-model fluxes) ──────────────────
BAND_PIVOT_UM = {              # observed-frame pivot wavelength [um]
    'best.MCam_u': 0.381,  'best.subaru.suprime.IB427': 0.426, 'best.SUBARU_B': 0.446,
    'best.subaru.suprime.IB464': 0.464, 'best.MCam_g': 0.487, 'best.subaru.suprime.V': 0.548,
    'best.subaru.suprime.r': 0.629, 'best.SUBARU_i': 0.768,
    'best.vista.vircam.J': 1.254, 'best.vista.vircam.H': 1.646, 'best.vista.vircam.Ks': 2.149,
    'best.IRAC1': 3.557, 'best.IRAC2': 4.504, 'best.IRAC3': 5.738, 'best.IRAC4': 7.927,
    'best.MIPS1': 23.68, 'best.PACS_green': 100.0, 'best.PSW': 250.0, 'best.ALMA6': 1250.0,
}

OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue                       # CEERS: observed side lives in the cis100 notebook
    wl = np.array([BAND_PIVOT_UM[b] for b in BAND_PIVOT_UM])
    f_ = np.array([float(T['row'][b]) for b in BAND_PIVOT_UM])
    OBS[tid] = dict(wl=wl, fnu=f_)     # mJy, observed frame

# optional: real observed photometry with errors, once staged on the cluster
if OBS_PHOT_HOOK and os.path.exists(OBS_PHOT_HOOK):
    print('observations.fits hook found — wire the per-band errors in here')
print(f'observed side ready for {len(OBS)} ALMA-C11 targets '
      f'({len(BAND_PIVOT_UM)} bands, 0.38 um - 1.25 mm observed)')


## Part 2 · mock stacks per target — dust_on vs observed

Median + 16-84% band of the member analogues' `dust_on` SEDs, placed at the target redshift,
against the CIGALE best-model fluxes. AGN/non-AGN split via the per-target `AGN_CLASS`.


In [ ]:
# ── Part 2 · per-target SED stacks ───────────────────────────────────────────
def stack_curves(rows, arm, sightline='mean'):
    curves = []
    for r in rows:
        sed = read_sed(arm, r['SNAPSHOT'], r['GROUPID_SNAPSHOT'], sightline=sightline)
        if sed is not None:
            curves.append(on_grid(*sed))
    C_ = np.asarray(curves)
    if C_.size == 0:
        return None
    return dict(med=np.nanmedian(C_, axis=0), lo=np.nanpercentile(C_, 16, axis=0),
                hi=np.nanpercentile(C_, 84, axis=0), n=len(C_), curves=C_)

for tid in [t for t in TARGETS if t in SELECTED and t in OBS]:
    A = SELECTED[tid]
    A_cls = _as_str(A['AGN_CLASS'])
    z = TARGETS[tid]['z']
    WOBS, FAC = obs_fnu_factor(z)
    fig, ax = plt.subplots(figsize=(8.5, 6))
    for cls in ('no_AGN', 'AGN'):
        S = stack_curves(A[A_cls == cls], 'dust_on')
        if S is None:
            continue
        ax.fill_between(WOBS, S['lo'] * FAC, S['hi'] * FAC, color=COL[cls], alpha=0.25, lw=0)
        ax.plot(WOBS, S['med'] * FAC, color=COL[cls], lw=2,
                label=f"SIMBA {'AGN host' if cls == 'AGN' else 'non-AGN host'} (N={S['n']})")
    ax.plot(OBS[tid]['wl'], OBS[tid]['fnu'], 'o', ms=6, mfc='w', mec=COL['obs'],
            label='CIGALE best-model fluxes', zorder=5)
    ax.set(xscale='log', yscale='log', xlim=(0.2, 2000),
           xlabel=r'observed-frame wavelength [$\mu$m]', ylabel=r'$F_\nu$ [mJy]',
           title=f"{tid} (z={z:.3f}) vs SIMBA analogues — dust_on")
    fin = OBS[tid]['fnu'][OBS[tid]['fnu'] > 0]
    ax.set_ylim(fin.min() / 3e2, fin.max() * 3e2)
    ax.grid(alpha=0.2, which='both', lw=0.5)
    ax.legend(fontsize=9, frameon=False, loc='lower center')
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, f'sed_stack_{_tid_safe(tid)}.png'),
                dpi=180, bbox_inches='tight')
    plt.show()


## Part 4 · torus MIR strength vs CLUMPY inclination

AGN hosts only. Per galaxy the torus contribution is the residual `nenkova_iXX − dust_on`
(same galaxy, same dust, only the injected template differs; dust_on at `findx=0`). Median
residuals for i=90/60/30, plus the ratio-to-dust_on version. Optional: rescale each residual by
`L_AGN,obs / L_AGN,injected` (needs the AGN-prop FITS staged — see header).


In [ ]:
# ── Part 4 · torus residual vs inclination ───────────────────────────────────
_agn_rows = UNION[np.asarray(UNION['AGN_ANY'], bool)]
print(f'AGN hosts with torus arms: {len(_agn_rows)}')

RES = {}
for arm in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30'):
    res, ratio = [], []
    for r in _agn_rows:
        snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
        on = read_sed('dust_on', snap, gid, sightline='findx0')
        nk = read_sed(arm, snap, gid, sightline='mean')   # mean handles the reused 4-sightline i90
        if on is None or nk is None:
            continue
        g_on, g_nk = on_grid(*on), on_grid(*nk)
        res.append(g_nk - g_on)
        with np.errstate(invalid='ignore', divide='ignore'):
            ratio.append(g_nk / g_on)
    if res:
        RES[arm] = dict(res=np.asarray(res), ratio=np.asarray(ratio), n=len(res))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for arm, D in RES.items():
    axes[0].plot(GRID, np.nanmedian(D['res'], axis=0), color=COL[arm], lw=2,
                 label=f"{arm} (N={D['n']})")
    axes[1].plot(GRID, np.nanmedian(D['ratio'], axis=0), color=COL[arm], lw=2)
axes[0].set(xscale='log', yscale='log', xlim=(0.1, 500), ylabel=r'$\nu L_\nu$ [erg s$^{-1}$]',
            xlabel=r'rest-frame wavelength [$\mu$m]',
            title='median torus residual (nenkova $-$ dust_on)')
axes[1].axhline(1, color='0.5', lw=0.8)
axes[1].set(xscale='log', yscale='log', xlabel=r'rest-frame wavelength [$\mu$m]',
            ylabel='nenkova / dust_on', title='median boost over the host')
axes[0].legend(fontsize=9, frameon=False)
for ax in axes:
    ax.grid(alpha=0.2, which='both', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'torus_vs_inclination.png'), dpi=180)
plt.show()

# optional L_AGN rescaling: residual * (L_AGN_obs / L_AGN_injected), L_AGN_injected = LBOL (0.1 Mdot c^2)
if os.path.exists(AGN_PROP_HOOK):
    print('AGN-prop FITS found — add the matched-L_AGN panel here')
else:
    print(f'AGN-prop FITS not staged ({AGN_PROP_HOOK}) — matched-L_AGN panel skipped')


## Part 5 · dust masses — sanitized CSV column vs simulated analogues

The CSV `M_dust [Msun]` column has **mixed units across rows** (some raw Msun, some scaled).
Adjudicate row-by-row against `Mdust_sc` (+err) and `Mdust_mbb`; adopt `Mdust_sc` where the raw
column is inconsistent, and print the adjudication so nothing is silent.


In [ ]:
# ── Part 5 · dust-mass comparison ────────────────────────────────────────────
print(f"{'tid':8s} {'M_dust[raw]':>12s} {'Mdust_sc':>12s} {'Mdust_mbb':>12s} {'adopted logMd':>13s}")
LOGMD_OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue
    raw, sc, mbb = (float(T['row'].get(k_, np.nan)) for k_ in
                    ('M_dust [Msun]', 'Mdust_sc', 'Mdust_mbb'))
    # raw is trustworthy only when it agrees with Mdust_sc within a factor of a few
    adopted = raw if (np.isfinite(raw) and np.isfinite(sc) and 0.3 < raw / sc < 3) else sc
    LOGMD_OBS[tid] = np.log10(adopted) if np.isfinite(adopted) and adopted > 0 else np.nan
    flag = '' if adopted is raw else '   <- raw inconsistent, using Mdust_sc'
    print(f'{tid:8s} {raw:12.3e} {sc:12.3e} {mbb:12.3e} {LOGMD_OBS[tid]:13.2f}{flag}')

_tids = [t for t in LOGMD_OBS if t in SELECTED]
fig, ax = plt.subplots(figsize=(9, 5))
for x, tid in enumerate(_tids):
    y = np.asarray(SELECTED[tid]['LOG_MDUST'], float)
    y = y[np.isfinite(y)]
    ax.plot(np.full(y.size, x) + np.random.uniform(-0.15, 0.15, y.size), y, 'o', ms=3,
            color='tab:blue', alpha=0.4, mec='none')
    if y.size:
        ax.hlines(np.median(y), x - 0.25, x + 0.25, color='tab:blue', lw=2)
    ax.plot(x, LOGMD_OBS[tid], 's', ms=7, color='k')
ax.set(xticks=range(len(_tids)), xticklabels=_tids,
       ylabel=r'$\log\,M_{\rm dust}\,[M_\odot]$',
       title='simulated analogue dust masses (blue) vs ALMA-C11 (black)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'dust_mass_almac11.png'), dpi=180)
plt.show()


## Part 6 · does the observed SED need an AGN? does it need to be obscured?

Model ladder per target, all against the CIGALE best-model fluxes: `dust_on` non-AGN hosts,
`dust_on` AGN hosts (**no AGN light** — `BH_SED=False`, so this isolates the host of an AGN),
then `nenkova_i30/i60/i90` = same AGN hosts **plus** the CLUMPY-transmitted AGN
(i=30 ≈ face-on/unobscured, i=90 = edge-on/obscured). Small-multiple SEDs, then band-level
residuals `Δlog F = log10(model stack / obs)` aggregated into rest-frame regimes.
Read-off: if only a Nenkova arm closes the MIR, SIMBA needs the AGN; the inclination that
minimises |Δlog| in the MIR says how obscured it must be. (i30 rows appear once that arm's
RT is run and the source map re-censused.)

In [ ]:
# ── Part 6 · AGN necessity + obscuration, against the observed bands ─────────
LADDER = (('host, non-AGN', 'dust_on', 'no_AGN', 'tab:blue'),
          ('host, AGN (no AGN light)', 'dust_on', 'AGN', 'tab:orange'),
          ('AGN i=30 (unobscured)', 'nenkova_i30', 'AGN', COL['nenkova_i30']),
          ('AGN i=60', 'nenkova_i60', 'AGN', COL['nenkova_i60']),
          ('AGN i=90 (obscured)', 'nenkova_i90', 'AGN', COL['nenkova_i90']))
REGIMES = (('opt <1um', 0.0, 1.0), ('NIR 1-3um', 1.0, 3.0),
           ('MIR 3-30um', 3.0, 30.0), ('FIR >30um', 30.0, np.inf))

_tids6 = [t for t in TARGETS if t in SELECTED and t in OBS]
DLOG = {lab: {rn: [] for rn, _, _ in REGIMES} for lab, _, _, _ in LADDER}

ncol6 = 4
nrow6 = int(np.ceil(len(_tids6) / ncol6))
fig, axes = plt.subplots(nrow6, ncol6, figsize=(4.4 * ncol6, 3.3 * nrow6))
for ax, tid in zip(np.ravel(axes), _tids6):
    A = SELECTED[tid]
    A_cls = _as_str(A['AGN_CLASS'])
    z = TARGETS[tid]['z']
    WOBS, FAC = obs_fnu_factor(z)
    ok = OBS[tid]['fnu'] > 0
    wl_o, f_o = OBS[tid]['wl'][ok], OBS[tid]['fnu'][ok]
    for lab, arm, cls, c_ in LADDER:
        S = stack_curves(A[A_cls == cls], arm)
        if S is None:
            continue
        mod = S['med'] * FAC
        fin = np.isfinite(mod) & (mod > 0)
        ax.plot(WOBS[fin], mod[fin], color=c_, lw=1.3,
                label=f'{lab} (N={S["n"]})' if tid == _tids6[0] else None)
        mod_at = 10 ** np.interp(np.log10(wl_o), np.log10(WOBS[fin]), np.log10(mod[fin]))
        d = np.log10(mod_at / f_o)
        rest = wl_o / (1.0 + z)
        for rn, lo, hi in REGIMES:
            DLOG[lab][rn].extend(d[(rest >= lo) & (rest < hi) & np.isfinite(d)])
    ax.plot(wl_o, f_o, 'o', ms=4, mfc='w', mec='k', zorder=5)
    ax.set(xscale='log', yscale='log', xlim=(0.2, 2000),
           ylim=(f_o.min() / 3e2, f_o.max() * 3e2), title=f'{tid} (z={z:.3f})')
    ax.grid(alpha=0.15, which='both', lw=0.4)
for ax in np.ravel(axes)[len(_tids6):]:
    ax.set_axis_off()
np.ravel(axes)[0].legend(fontsize=7, frameon=False, loc='lower center')
fig.text(0.5, 0.002, r'observed-frame wavelength [$\mu$m]', ha='center')
fig.text(0.002, 0.5, r'$F_\nu$ [mJy]', va='center', rotation='vertical')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'agn_ladder_seds.png'), dpi=160, bbox_inches='tight')
plt.show()

# regime summary across all targets x bands
_lcol = {lab: c_ for lab, _, _, c_ in LADDER}
labs = [lab for lab, _, _, _ in LADDER if any(len(DLOG[lab][rn]) for rn, _, _ in REGIMES)]
xs = np.arange(len(REGIMES))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for k, lab in enumerate(labs):
    off = (k - (len(labs) - 1) / 2) * 0.15
    ab = [np.nanmedian(np.abs(DLOG[lab][rn])) if len(DLOG[lab][rn]) else np.nan
          for rn, _, _ in REGIMES]
    sg = [np.nanmedian(DLOG[lab][rn]) if len(DLOG[lab][rn]) else np.nan for rn, _, _ in REGIMES]
    axes[0].bar(xs + off, ab, 0.15, color=_lcol[lab], label=lab)
    axes[1].bar(xs + off, sg, 0.15, color=_lcol[lab])
axes[1].axhline(0, color='0.4', lw=0.8)
for ax, t_ in zip(axes, (r'median $|\Delta\log F|$ (accuracy)',
                         r'median $\Delta\log F$ (bias: model $-$ obs)')):
    ax.set(xticks=xs, xticklabels=[rn for rn, _, _ in REGIMES], title=t_)
    ax.grid(alpha=0.2, axis='y', lw=0.5)
axes[0].set_ylabel('dex')
axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'agn_ladder_residuals.png'), dpi=180, bbox_inches='tight')
plt.show()

print('median dlog10(model/obs) per regime  [negative = model too faint]')
print(f'{"model":26s} ' + ' '.join(f'{rn:>12s}' for rn, _, _ in REGIMES))
for lab in labs:
    v = ' '.join(f'{(np.nanmedian(DLOG[lab][rn]) if len(DLOG[lab][rn]) else np.nan):12.2f}'
                 for rn, _, _ in REGIMES)
    print(f'{lab:26s} {v}')

## Part 7 · if the dust ratios match, does the attenuation match?

The analogues were *selected* in the age–mass windows where SIMBA predicts dusty sources
(dust-to-molecular ratio > 10⁻³, controls < 10⁻⁴). Here: per-galaxy `A_V` (from Part 3's
dust_on/dust_off difference, `findx0`) against `log M_dust/M*` and `log M_dust/M_gas`,
with the observed sources overplotted (`Av_ISM` vs the sanitized dust masses of Part 5,
and vs the CSV `logDGR`). Caveat: the selection tables carry **total** gas, not H₂, so the
sim x-axis of the right panel is a lower bound on the dust-to-molecular ratio; the dashed
and dotted guides are the 10⁻³ / 10⁻⁴ selection thresholds. Depends on Part 5 (`LOGMD_OBS`). Part 9d redoes the figure like-for-like with the mock CIGALE `Av_ISM` (needs the Part 9 fits).


In [ ]:
# ── Part 7 · dust ratios vs attenuation ──────────────────────────────────────
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

AV_ALL = {}                                # (snap,gid) -> A_V at findx0, union-wide
for r in UNION:
    k = (int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))
    c_ = attenuation_curve(*k)
    if c_ is not None and np.isfinite(c_[_jV]):
        AV_ALL[k] = float(c_[_jV])
print(f'A_V available for {len(AV_ALL)}/{len(UNION)} union galaxies (needs dust_on+dust_off)')

_rows7 = {}
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows7.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
ds, dg, av, isagn = [], [], [], []
for k, r in _rows7.items():
    if k not in AV_ALL:
        continue
    md, mg = float(r['MDUST']), float(r['MGAS'])
    ds.append(np.log10(md) - float(r['LOG_MSTAR']) if md > 0 else np.nan)
    dg.append(np.log10(md / mg) if (md > 0 and mg > 0) else np.nan)
    av.append(AV_ALL[k])
    isagn.append(str(r['AGN_CLASS']) == 'AGN')
ds, dg, av, isagn = map(np.asarray, (ds, dg, av, isagn))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)
for flag, c_, lab in ((False, 'tab:blue', 'non-AGN host'), (True, 'tab:orange', 'AGN host')):
    m = isagn == flag
    axes[0].plot(ds[m], av[m], 'o', ms=4, color=c_, alpha=0.55, mec='none', label=lab)
    axes[1].plot(dg[m], av[m], 'o', ms=4, color=c_, alpha=0.55, mec='none')
for tid, T in TARGETS.items():             # observed side
    if T['row'] is None or not np.isfinite(T['av_ism']):
        continue
    if tid in LOGMD_OBS and np.isfinite(LOGMD_OBS[tid]):
        axes[0].errorbar(LOGMD_OBS[tid] - T['logm'], T['av_ism'], yerr=T['av_ism_err'],
                         fmt='s', ms=6, color='k', capsize=2, zorder=5)
    dgr = float(T['row'].get('logDGR', np.nan))
    if np.isfinite(dgr):
        axes[1].errorbar(dgr, T['av_ism'], yerr=T['av_ism_err'], fmt='s', ms=6,
                         color='k', capsize=2, zorder=5)
axes[1].axvline(-3, color='0.6', ls='--', lw=0.8)      # dusty-selection threshold
axes[1].axvline(-4, color='0.6', ls=':', lw=0.8)       # control threshold
axes[0].set_xlim(-5)
axes[1].set_xlim(-4)
axes[0].set(xlabel=r'$\log\,M_{\rm dust}/M_\star$',
            ylabel=r'$A_V$ [mag] (sim)  /  Av_ISM (obs, black)')
axes[1].set(xlabel=r'sim: $\log\,M_{\rm dust}/M_{\rm gas,tot}$   ·   '
                   r'obs: logDGR ($M_{\rm dust}/M_{\rm mol}$)')
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle('does matching the dust ratios reproduce the attenuation?')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dust_ratio_vs_av.png'), dpi=180, bbox_inches='tight')
plt.show()

for name, x in (('log Md/M*', ds), ('log Md/Mgas', dg)):
    m = np.isfinite(x) & np.isfinite(av)
    if spearmanr is not None and m.sum() > 3:
        rho, p = spearmanr(x[m], av[m])
        print(f'Spearman A_V vs {name}: rho={rho:+.2f} (p={p:.1e}, N={m.sum()})')

## Part 8 · host viewing angle — does attenuation need a particular inclination?

The dust arms carry 4 quasi-orthogonal sightlines (`THETA=[0,45,90,135]`). Per galaxy:
`A_V` per sightline (matched dust_on/dust_off index). Left: sightline-to-sightline spread vs
the median. Right, per target: the gap `Av_ISM(obs) − median sim A_V` (grey bars) against the
typical ± half sightline range of its members (red) — if the red bars are much shorter than
the grey ones, no viewing angle reconciles sim and observed attenuation, and the dust
content/geometry itself must differ. (Torus inclination is Part 6's axis, not this one.)
Depends on Part 7 (`AV_ALL`).
A third figure plots each member's $\log M_{\rm dust}$ against its $A_V$ **split by the
four sightlines** (colored by $\theta$): does more dust mean more attenuation regardless
of viewing angle, or does geometry scatter the relation? `AV_SIGHT` keeps the
sightline-indexed vector (NaN where a sightline's flux ratio is unusable), so the colors
identify the same $\theta$ across galaxies. Also depends on Part 7's `spearmanr`.


In [ ]:
# ── Part 8 · A_V across the 4 dust-arm sightlines ────────────────────────────
def _sed_allsight(arm, snap, gid):
    p = rtout_path(arm, snap, gid)
    if p is None:
        return None
    sed = ModelOutput(p).get_sed(inclination='all', aperture=-1)
    wav = np.asarray(sed.wav, float)
    val = np.atleast_2d(np.asarray(sed.val, float))
    s_ = np.argsort(wav)
    return wav[s_], val[:, s_]

AV_SIGHT = {}                              # (snap,gid) -> A_V per sightline
for k in AV_ALL:
    on, off = _sed_allsight('dust_on', *k), _sed_allsight('dust_off', *k)
    if on is None or off is None:
        continue
    n = min(on[1].shape[0], off[1].shape[0])
    with np.errstate(invalid='ignore', divide='ignore'):
        v = np.array([(-2.5 * np.log10(on_grid(on[0], on[1][s_]) /
                                       on_grid(off[0], off[1][s_])))[_jV] for s_ in range(n)])
    if np.isfinite(v).sum() >= 2:
        AV_SIGHT[k] = v                    # sightline-indexed, may hold NaN

med8 = np.array([np.nanmedian(v) for v in AV_SIGHT.values()])
rng8 = np.array([np.nanmax(v) - np.nanmin(v) for v in AV_SIGHT.values()])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].plot(med8, rng8, 'o', ms=4, color='tab:blue', alpha=0.6, mec='none')
axes[0].set(xlabel=r'median $A_V$ across sightlines [mag]',
            ylabel=r'max$-$min $A_V$ [mag]',
            title=f'sightline spread per galaxy (N={len(AV_SIGHT)})')

_t8 = [t for t in TARGETS if TARGETS[t]['sample'] == 'almac11'
       and np.isfinite(TARGETS[t]['av_ism']) and t in SELECTED]
gaps, sprd = [], []
for tid in _t8:
    ks = [(int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])) for r in SELECTED[tid]]
    vs = [AV_SIGHT[k] for k in ks if k in AV_SIGHT]
    if not vs:
        gaps.append(np.nan)
        sprd.append(np.nan)
        continue
    gaps.append(TARGETS[tid]['av_ism'] - np.median([np.nanmedian(v) for v in vs]))
    sprd.append(np.median([(np.nanmax(v) - np.nanmin(v)) / 2 for v in vs]))
gaps, sprd = np.asarray(gaps), np.asarray(sprd)
x8 = np.arange(len(_t8))
axes[1].bar(x8, gaps, 0.6, color='0.75', label=r'Av_ISM(obs) $-$ median sim $A_V$')
axes[1].errorbar(x8, np.zeros(len(_t8)), yerr=sprd, fmt='none', ecolor='tab:red',
                 capsize=2, label=r'$\pm$ half sightline range')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set(xticks=x8, xticklabels=_t8, ylabel='[mag]',
            title='can viewing angle bridge the gap?')
axes[1].tick_params(axis='x', rotation=60)
axes[1].legend(fontsize=8, frameon=False)
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'sightline_av_spread.png'), dpi=180, bbox_inches='tight')
plt.show()
print(f'median sightline A_V spread (max-min): {np.median(rng8):.3f} mag | '
      f'median |obs - sim| A_V gap: {np.nanmedian(np.abs(gaps)):.2f} mag')

# ── dust mass vs attenuation, split by the 4 sightlines ──────────────────────
THETA8 = (0, 45, 90, 135)
_rows8 = {}                                # (snap,gid) -> selection row, deduped
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows8.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
fig, ax = plt.subplots(figsize=(7, 5.2))
for s_ in range(4):
    x_, y_ = [], []
    for k, r in _rows8.items():
        v = AV_SIGHT.get(k)
        if v is None or s_ >= v.size or not np.isfinite(v[s_]):
            continue
        md = float(r['MDUST'])
        if md > 0:
            x_.append(np.log10(md))
            y_.append(float(v[s_]))
    x_, y_ = np.asarray(x_), np.asarray(y_)
    ax.plot(x_, y_, 'o', ms=4, color=f'C{s_}', alpha=0.55, mec='none',
            label=fr'$\theta={THETA8[s_]}^\circ$')
    if spearmanr is not None and x_.size > 3:
        rho, p = spearmanr(x_, y_)
        print(f'theta={THETA8[s_]:3d}: Spearman A_V vs log Mdust '
              f'rho={rho:+.2f} (p={p:.1e}, N={x_.size})')
ax.set(xlabel=r'$\log\,M_{\rm dust}$ [$M_\odot$]', ylabel=r'$A_V$ [mag]',
       title='dust mass vs attenuation per sightline')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=9, frameon=False, title='sightline')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dustmass_vs_av_sightlines.png'), dpi=180)
plt.show()


## Part 9 · consistent A_V — CIGALE fits of the mock dust_on photometry

Part 3 compares two **different quantities**: the RT differential
$A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ (emergent, *total* effective attenuation) against
the observed `bayes.attenuation.Av_ISM` — a **parameter** of the modified Charlot & Fall
model, and only its **ISM component** (with the CF00 default $\mu=0.44$ the young populations
see $A_V^{\rm ISM}/\mu \approx 2.3\times$ more). Part of the Part 3 discrepancy is therefore
definitional. This part closes the loop by pushing the mocks through **the same estimator as
the observations**:

- **9a** — observed-frame photometry of every union galaxy's `dust_on` SED (`findx0`,
  total aperture) in **exactly the 19 bands** of the observed CIGALE fit (MegaCam u/g,
  Suprime B/V/r/i + IB427/464, VISTA JHKs, IRAC 1–4, MIPS 24, PACS 100, SPIRE 250,
  ALMA band 6), with the Hyperion MC errors propagated. Source-map driven (the dust_on arm
  spans the new `sed_almac11` tree **and** two reused CEERS trees), reusing the
  `MakeSED.extract_flux_batch` primitives.
- **9b** — CIGALE 2025 inputs + run dirs (**one per snapshot × metallicity group**, so the
  `age_main` grid can respect $t_{\rm universe}(z)$ — the m25 7e convention) + **one SLURM
  job array**. Module chain: `sfhdelayed` (star-forming analogues) + `bc03` (Chabrier) +
  `nebular` + `dustatt_modified_CF00` + `dl2014` + rest-frame UVJ — module-for-module what
  the observed fit used (its CSV carries `Av_ISM`, `V_B90`, `dust.umean`). All 19 band names
  verified against the **installed cluster DB** (2025.1, custom `alma.band6` already
  registered). **SIMBA-truth priors** (the m25 `split_by_metallicity` machinery): caesar
  stellar Z → one `bc03` node per run, SFR-weighted gas Z → ≤3 `nebular` `zgas` nodes,
  and ≤3 age-quantile sub-runs per Z group, each `age_main` grid floored at its bin's
  youngest mass-weighted age. $M_*$ stays free — it is the fitted normalization, not a
  grid axis.
- **9c** — after the array drains: (i) the Part 3 figure **like-for-like** — mock
  `bayes.attenuation.Av_ISM` vs observed `Av_ISM`; (ii) the **estimator-closure** test only a
  simulation can do — CIGALE's total V attenuation `attenuation.generic.bessell.V` (the
  stock-DB name of the observed `V_B90` column) and `Av_ISM` against the RT truth per galaxy,
  i.e. the bias of the estimator itself; (iii) **truth recovery** — recovered $\log M_*$ and
  mass-weighted age vs caesar, plus the `Av_ISM` shift against the archived prior-free runs.

**Run order.** 9a + 9b in the cluster kernel (9a reads ~480 rtouts, ≈10–15 min) →
`sbatch` the printed job (one task per snapshot × Z group, minutes each) → 9c (needs Part 3
in-session for `attenuation_curve`).

**Caveats.** The priors deliberately break strict estimator symmetry with the observed fit
(which had no Z/age truth): the prior-informed fit measures the attenuation bias with the
stellar population pinned, isolating the dust side of the degeneracy. The original
prior-free runs (`dust_on_findx0_snapNNN`, no `_Zs` suffix) stay on disk as the baseline —
9c auto-detects the Z-tagged runs and prints the shift between the two. Single sightline
`findx0` (Part 3 convention — Part 8 shows the sightline spread is small); NaN MC errors are
repaired to 10% of the flux and CIGALE adds `additionalerror = 0.1` in quadrature at fit
time.

In [ ]:
# ── Part 9a · mock photometry: observed-frame fluxes in the 19 ALMA-C11 bands ─
# Source-map driven: the dust_on arm spans 3 RT trees (sed_almac11 + 2 reused
# CEERS trees), so MakeSED.extract_flux_batch's fixed snap_XXX/gal_X path
# convention cannot be pointed at one tree — this loop reuses its exact
# primitives instead (same unit conversion + MC-error propagation).
# Needs only Part 0.
from simbanator.sed.makesed import _read_sed, _sed_to_mJy
from simbanator.sed.flux_extraction import (get_svo_filters, load_local_filters,
                                            flux_extraction)

CIGDIR = os.path.join(OUTDIR, 'cigale')
os.makedirs(CIGDIR, exist_ok=True)
SIGHTLINE_FINDX = 0            # findx0 — same convention as the Part 3 A_V

# the 19 bands of the observed CIGALE fit; SVO fetch per instrument. The
# Suprime narrowbands are deliberately NOT fetched: at 2500 SED points the
# IB427/IB464 filters get ~8 native points across (enough for the trapezoid
# convolution) but the ~7 nm NB ones would be under-sampled.
SVO_SETS = [
    ('Subaru',   'Suprime', ['B', 'V', 'r', 'i', 'IB427', 'IB464']),
    ('CFHT',     'MegaCam', ['u', 'g']),
    ('Paranal',  'VIRCAM',  ['J', 'H', 'Ks']),
    ('Spitzer',  'IRAC',    ['I1', 'I2', 'I3', 'I4']),
    ('Spitzer',  'MIPS',    ['24mu']),
    ('Herschel', 'PACS',    ['green']),
    ('Herschel', 'SPIRE',   ['PSW']),
]
LOCAL_FILTERS = {   # custom top-hat; its CIGALE twin 'alma.band6' is registered
    'ALMA': {'ALMA': {'band6': os.path.join(HOME, 'ALMA_band6.res')}},
}
profiles = {}
for _fac, _inst, _filts in SVO_SETS:
    for f_, d_ in get_svo_filters(_fac, _inst, filters=_filts,
                                  wave_unit='micron').items():
        for i_, fd_ in d_.items():
            profiles.setdefault(f_, {}).setdefault(i_, {}).update(fd_)
for f_, d_ in load_local_filters(LOCAL_FILTERS, 'micron').items():
    for i_, fd_ in d_.items():
        profiles.setdefault(f_, {}).setdefault(i_, {}).update(fd_)
_nfilt = sum(len(fd_) for d_ in profiles.values() for fd_ in d_.values())
print(f'{_nfilt} filter profiles ready (expect 19)')

# snap -> z (identical for every member of a snapshot)
ZSNAP = {int(r['SNAPSHOT']): float(r['REDSHIFT'])
         for t_ in SELECTED.values() for r in t_}

rows9, skip9, fail9 = [], [], []
for kk, r in enumerate(UNION):
    snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
    p = rtout_path('dust_on', snap, gid)
    if p is None:
        skip9.append((snap, gid)); continue
    try:
        wav_raw, flux_raw, unc_raw = _read_sed(p, aperture=-1,
                                               uncertainties=True)
        z = ZSNAP[snap]
        wav, flux = _sed_to_mJy(wav_raw, flux_raw, z, apply_redshift=True)
        fx = flux[SIGHTLINE_FINDX] if flux.ndim == 2 else flux
        ux = None
        if unc_raw is not None:
            _, unc = _sed_to_mJy(wav_raw, unc_raw, z, apply_redshift=True)
            ux = unc[SIGHTLINE_FINDX] if unc.ndim == 2 else unc
        res9 = flux_extraction(None, None, wav, fx, wave_unit='micron',
                               filter_list=profiles, flux_unc=ux)
    except Exception as e:
        fail9.append((snap, gid, str(e))); continue
    row = {'gal_id_at_snap': gid, 'snap': snap, 'redshift': z}
    for fac_, inst_d in res9.items():
        for inst_, filt_d in inst_d.items():
            for fn_, fd_ in filt_d.items():
                col = f'{fac_}.{inst_}.{fn_}'
                row[col] = fd_['mJy']
                row[f'{col}_err'] = fd_.get('mJy_err', np.nan)
    rows9.append(row)
    if (kk + 1) % 50 == 0:
        print(f'  {kk + 1}/{len(UNION)} rtouts read')

FLUX_FITS9 = os.path.join(CIGDIR,
                          f'almac11_fluxes_dust_on_findx{SIGHTLINE_FINDX}.fits')
Table(rows9).write(FLUX_FITS9, overwrite=True)
nb9 = (len(rows9[0]) - 3) // 2 if rows9 else 0
print(f'{len(rows9)} galaxies x {nb9} bands -> {FLUX_FITS9}')
print(f'skipped (RT incomplete): {len(skip9)} | read failures: {len(fail9)}')
for s_ in fail9[:10]:
    print('   fail:', s_)

### Part 9b — CIGALE run dirs (snapshot × Z-group) + ONE SLURM job array

CIGALE never fits inside this kernel: the cell only writes the data files, every
`pcigale.ini` (+`.spec`, via `simbanator.sed.cigale.prepare_run` — no `pcigale
init`/`genconf` by hand) and one sbatch array with **one task per (snapshot, metallicity
group) run**. Grid choices:

- **`sfhdelayed`**, not the quench-tuned `sfhdelayedbq` of the m25 runs — the analogues are
  star-forming; `age_main` is filtered per snapshot to $< 0.95\,t_{\rm universe}(z)$.
- **SIMBA-truth priors** (`USE_SIMBA_PRIORS`, the m25 7e machinery — closes the old
  "no Z prior" caveat): per galaxy the caesar `metallicities.stellar` is snapped to one
  `bc03` node and objects sharing a node form one run (`cg.split_by_metallicity`; run dirs
  `dust_on_findx0_snapNNN_Zs0p008 | _Zs0p02 | … | _Zsfree`); the SFR-weighted gas Z
  restricts the `nebular` `zgas` grid to ≤3 nodes; and each Z group is sub-split into ≤3
  **age-quantile bins** (run dirs get an extra `_aXpX` tag), each run's `age_main` grid
  floored at the bin's *youngest* mass-weighted age — the bound is per galaxy
  (mass-weighted age < `age_main` for any SFH), so a shared grid can only be floored at
  its youngest member; narrow bins keep that floor tight for the old members. This removes
  the young+dusty corner of the age–attenuation degeneracy, exactly where `Av_ISM` leaks —
  and lets the grid extend past the old `AGE_BASE` cap of 6 Gyr, below the true 7.7 Gyr
  ages at snaps 128–132. $M_*$ cannot be a grid prior — CIGALE fits it as the model
  normalization — so mass is the truth-recovery check in 9c instead.
- **`dustatt_modified_CF00`** with `Av_ISM` up to 3 mag — same module as the observed fit, so
  `bayes.attenuation.Av_ISM` is directly comparable, and `attenuation.generic.bessell.V`
  (total emergent V attenuation, the observed `V_B90`) comes for free from the module's
  default `filters` entry. The attenuation grid is **not** prior-informed — it is the
  measurand.
- **`fit_bands`** pins the fit to the 19 observed bands; everything is still predicted.
- estimated variables include `stellar.metallicity` (the per-run pin, echoed back), the UVJ
  rest colours (observed `UV`/`VJ` columns) and `dust.mass`/`dust.luminosity` (cross-check
  of Part 5 with fitted rather than catalog dust masses).

Submit with the printed `sbatch` line; re-running the cell after edits just rewrites the run
dirs (CIGALE keeps old `out/` as `<timestamp>_out/`). Set `SKIP_IF_DONE=True` to make
resubmits skip finished runs. The pre-prior run dirs (`dust_on_findx0_snapNNN`, no `_Zs`
suffix) are left untouched — 9c uses them as the prior-free baseline.

In [ ]:
# ── Part 9b · CIGALE inputs + run dirs (snapshot x Z-group) + SLURM array ────
# Self-contained after Part 0 + the Part 9a FITS on disk (kernel-restart safe).
# SIMBA-truth priors (the m25 Part 7e machinery): the caesar stellar Z pins ONE
# bc03 metallicity node per run (cg.split_by_metallicity), the SFR-weighted gas
# Z restricts the nebular zgas grid (<= 3 nodes), and each run's oldest mass-
# weighted stellar ages set LOWER bounds on the age_main grid (the mass-
# weighted age is < age_main for ANY SFH): each Z group is sub-split into <=3
# age-quantile bins, one run each, floored at the bin's YOUNGEST member. M*
# cannot be a grid prior — CIGALE fits it as the model normalization — so
# mass stays a recovery check in 9c.
from simbanator.sed import cigale as cg
from simbanator.io.simba import Simulation

PCIGALE_CMD    = cg.find_pcigale()   # dedicated conda env; no activation needed
CORES_PER_TASK = 8
PLOT_SEDS      = True
SKIP_IF_DONE   = False
USE_SIMBA_PRIORS = True              # False -> the original prior-free 9b runs
print('pcigale:', PCIGALE_CMD)

CIGDIR     = os.path.join(OUTDIR, 'cigale')
CIGRUNS    = os.path.join(OUTDIR, 'cigale_runs')
FLUX_FITS9 = os.path.join(CIGDIR, 'almac11_fluxes_dust_on_findx0.fits')

# the 19 fitted bands = exactly the observed information content (CIGALE names
# verified against the installed 2025.1 DB; SVO 'r'/'i' map to CIGALE 'r+'/'i+')
FIT_BANDS = [
    'cfht.megacam.u', 'cfht.megacam.g',
    'subaru.suprime.B', 'subaru.suprime.V', 'subaru.suprime.r+',
    'subaru.suprime.i+', 'subaru.suprime.IB427', 'subaru.suprime.IB464',
    'paranal.vircam.J', 'paranal.vircam.H', 'paranal.vircam.Ks',
    'spitzer.irac.I1', 'spitzer.irac.I2', 'spitzer.irac.I3', 'spitzer.irac.I4',
    'spitzer.mips.24mu', 'herschel.pacs.green', 'herschel.spire.PSW',
    'alma.band6',
]

SED_MODULES = ('sfhdelayed', 'bc03', 'nebular', 'dustatt_modified_CF00',
               'dl2014', 'restframe_parameters', 'redshifting')
AGE_BASE = [750, 1000, 1500, 2000, 3000, 4000, 5000, 6000]        # Myr

def _module_params9(z, age_lo_myr=0.0):
    # age_main < 0.95 t_universe(z) in every grid combination (m25 7e rule),
    # AND >= the run members' oldest mass-weighted age (SIMBA truth bound)
    tmax = 0.95 * Planck13.age(z).to(u.Myr).value
    lo = min(float(age_lo_myr), 0.85 * tmax)   # keep a fittable window even
                                               # for members older than tmax
    ages = [a for a in AGE_BASE if lo <= a < tmax]
    if len(ages) < 3:                          # densify inside (lo, tmax)
        ages = sorted({int(round(a_))
                       for a_ in np.linspace(max(lo, 0.25 * tmax), tmax, 4)})
    return {
        'sfhdelayed': {'tau_main': [250, 500, 1000, 2000, 4000],
                       'age_main': ages,
                       'tau_burst': [50.0], 'age_burst': [20, 100],
                       'f_burst': [0.0, 0.05, 0.15]},
        'bc03': {'imf': 1, 'metallicity': [0.008, 0.02, 0.05]},
        'dustatt_modified_CF00': {
            'Av_ISM': [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]},
        'dl2014': {'qpah': [0.47, 2.5], 'umin': [1.0, 5.0, 10.0, 25.0],
                   'gamma': [0.02, 0.1]},
        'restframe_parameters': {
            'colours_filters': ('generic.johnson.U-generic.johnson.V & '
                                'generic.johnson.V-generic.johnson.J')},
    }

ANALYSIS_PARAMS9 = {
    'variables': ['stellar.m_star', 'stellar.age_m_star', 'stellar.metallicity',
                  'sfh.sfr', 'sfh.sfr100Myrs',
                  'sfh.tau_main', 'sfh.age_main', 'sfh.f_burst',
                  'attenuation.Av_ISM', 'attenuation.Av_BC',
                  'attenuation.generic.bessell.V',
                  'attenuation.generic.bessell.B',
                  'dust.luminosity', 'dust.mass',
                  'param.restframe_generic.johnson.U-generic.johnson.V',
                  'param.restframe_generic.johnson.V-generic.johnson.J'],
    'save_best_sed': True,
}

# full-catalog input file, then NaN-error repair, then one file per snapshot
data_all = cg.write_cigale_input(
    FLUX_FITS9, os.path.join(CIGDIR, 'cigale_dust_on_findx0.fits'))
t9 = Table.read(data_all)
_bands9 = [c for c in t9.colnames
           if c not in ('id', 'redshift', 'distance') and not c.endswith('_err')]
n_fix = 0
for b_ in _bands9:
    f_ = np.asarray(t9[b_], float)
    e_ = np.asarray(t9[f'{b_}_err'], float)
    bad = np.isfinite(f_) & ~np.isfinite(e_)
    if bad.any():
        t9[f'{b_}_err'][bad] = 0.1 * np.abs(f_[bad])
        n_fix += int(bad.sum())
if n_fix:
    t9.write(data_all, overwrite=True)
print(f'[sanitize] {n_fix} NaN error(s) -> 10% of the flux (CIGALE adds '
      f'additionalerror=0.1 in quadrature on top)')

_snap9 = np.array([int(re.match(r'snap(\d+)_', str(i_)).group(1))
                   for i_ in t9['id']])

# ── SIMBA truth per union galaxy: one caesar read per snapshot ──
# metallicities.stellar = mass-weighted stellar metal fraction (bc03 convention,
# 0.02 = solar); metallicities.sfr_weighted = the nebular-relevant gas Z;
# ages.mass_weighted in Gyr. Non-positive values (e.g. SFR=0 quiescents) -> no
# prior for that quantity (Zsfree group / default grids).
ZS_MAP, ZG_MAP, AGEMW_MAP = {}, {}, {}
if USE_SIMBA_PRIORS:
    _sim9 = Simulation('cis100')
    for snap in sorted(set(_snap9.tolist())):
        with h5py.File(_sim9.get_caesar_file(int(snap)), 'r') as fc_:
            d_ = fc_['galaxy_data']
            for g_, zs_, zg_, ag_ in zip(
                    np.asarray(d_['GroupID'][:], int),
                    np.asarray(d_['dicts/metallicities.stellar'][:], float),
                    np.asarray(d_['dicts/metallicities.sfr_weighted'][:], float),
                    np.asarray(d_['dicts/ages.mass_weighted'][:], float)):
                k_ = f'snap{int(snap):03d}_gal{int(g_)}'
                ZS_MAP[k_] = float(zs_) if zs_ > 0 else np.nan
                ZG_MAP[k_] = float(zg_) if zg_ > 0 else np.nan
                AGEMW_MAP[k_] = float(ag_) if ag_ > 0 else np.nan
    _in9 = [str(i_) for i_ in t9['id']]
    print(f'[priors] caesar truth for '
          f'{sum(np.isfinite(ZS_MAP.get(i_, np.nan)) for i_ in _in9)}/'
          f'{len(_in9)} galaxies (Z_star -> bc03 node, Z_gas^SFR -> zgas '
          f'grid, age_mw -> age_main lower bound)')

run_dirs9 = []
for snap in sorted(set(_snap9.tolist())):
    sub = t9[_snap9 == snap]
    z = float(np.median(np.asarray(sub['redshift'], float)))
    df = os.path.join(CIGDIR, f'cigale_dust_on_findx0_snap{snap:03d}.fits')
    sub.write(df, overwrite=True)
    if USE_SIMBA_PRIORS:   # one run per (snapshot, bc03 Z node); m25 7e pattern
        groups = cg.split_by_metallicity(df, ZS_MAP, zgas=ZG_MAP)
    else:
        groups = [{'tag': '', 'path': df, 'module_params': None}]
    for grp in groups:
        gt = Table.read(grp['path'])
        _ag = np.array([AGEMW_MAP.get(str(i_), np.nan)
                        for i_ in gt['id']], float)
        # sub-split each Z group by mass-weighted age: the floor of a SHARED
        # age_main grid can only be its youngest member's age_mw (the bound
        # is per galaxy), so one run per age-quantile bin keeps the floor
        # tight for the old members instead of wasting the prior on them
        # (snap 131 spans 2.3-7.6 Gyr in one Z group).
        n_bins = int(np.clip(np.isfinite(_ag).sum() // 8, 1, 3))
        edges = (np.nanquantile(_ag, np.linspace(0, 1, n_bins + 1))
                 if n_bins > 1 else None)
        for b_ in range(n_bins):
            if edges is None:
                m_ = np.ones(len(gt), bool)
            else:
                hi_ = edges[b_ + 1] if b_ < n_bins - 1 else np.inf
                m_ = (_ag >= edges[b_]) & (_ag < hi_)
                if b_ == 0:
                    m_ |= ~np.isfinite(_ag)      # no-age objects: no-floor bin
            if not m_.any():
                continue
            _agb = _ag[m_]
            age_lo = (1e3 * np.nanmin(_agb)
                      if np.isfinite(_agb).all() and _agb.size else 0.0)
            atag = (f'a{np.nanmin(_agb):.1f}'.replace('.', 'p')
                    if edges is not None else '')
            pth = (grp['path'].replace('.fits', f'_{atag}.fits')
                   if atag else grp['path'])
            if atag:
                gt[m_].write(pth, overwrite=True)
            mp = _module_params9(z, age_lo_myr=age_lo)
            for k_, v_ in (grp['module_params'] or {}).items():
                mp[k_] = {**mp.get(k_, {}), **v_}    # keep e.g. bc03's imf=1
            rd = os.path.join(
                CIGRUNS, f'dust_on_findx0_snap{snap:03d}'
                + (f"_{grp['tag']}" if grp['tag'] else '')
                + (f'_{atag}' if atag else ''))
            cg.prepare_run(rd, pth, sed_modules=SED_MODULES,
                           module_params=mp, analysis_params=ANALYSIS_PARAMS9,
                           cores=CORES_PER_TASK, fit_bands=FIT_BANDS)
            run_dirs9.append(rd)

job9 = cg.write_slurm_array(run_dirs9,
                            os.path.join(CIGRUNS, 'cigale_almac11.job'),
                            pcigale_cmd=PCIGALE_CMD, cores=CORES_PER_TASK,
                            plots=PLOT_SEDS, job_name='cigale_almac11',
                            skip_if_done=SKIP_IF_DONE)
print(f'\n{len(run_dirs9)} run dir(s) prepared. Submit with:\n'
      f'  sbatch {job9}\n'
      f'(add -p <partition> if needed; logs in '
      f'{os.path.join(CIGRUNS, "slurm_logs")})')

### Part 9c — after the fits: consistent Av_ISM comparison + estimator closure

Run once the array has drained (`ls cigale_runs/dust_on_findx0_snap*/out/results.fits`).
Needs Part 3 in-session (`attenuation_curve`). Auto-detects the prior-informed runs
(`*_Zs*` dirs) and falls back to the prior-free ones when none are fitted yet; when both
exist, the median `Av_ISM` shift between them is printed. Three figures + a per-target
table, all saved:

- **like-for-like** — the Part 3 strip plot redone with the mock `bayes.attenuation.Av_ISM`
  (green) against the observed `Av_ISM` (black); the RT $A_V$ medians (dashed blue) show how
  much of the original discrepancy was definitional.
- **estimator closure** — per galaxy, CIGALE's total V attenuation
  (`attenuation.generic.bessell.V` ≡ observed `V_B90`, orange) and `Av_ISM` (green) against
  the RT truth. The orange series answers "does CIGALE recover the true attenuation from 19
  bands?"; the green one shows what fraction the ISM component captures.
- **truth recovery** — recovered vs caesar $\log M_*$ and mass-weighted stellar age (from
  the `rt_union.fits` UNION columns). With Z pinned and the age floor set by the priors,
  any residual mass/age bias bounds how much of a remaining $A_V$ discrepancy can still be
  an SFH artefact.

The joined per-galaxy table lands in `cigale/almac11_cigale_av_comparison.fits`.

In [ ]:
# ── Part 9c · consistent A_V: CIGALE(mock) vs CIGALE(obs) vs RT truth ────────
# Run AFTER the job array drains. Needs Part 0 (attenuation_curve).
from simbanator.sed.cigale import nmad

CIGRUNS = os.path.join(OUTDIR, 'cigale_runs')
CIGDIR  = os.path.join(OUTDIR, 'cigale')
_rdirs = sorted(glob.glob(os.path.join(CIGRUNS, 'dust_on_findx0_snap*_Zs*')))
_prior_runs = bool(_rdirs)
if not _prior_runs:                     # fall back to the prior-free 9b runs
    _rdirs = sorted(glob.glob(os.path.join(
        CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]')))
CFIT, _miss9 = {}, []
for rd in _rdirs:
    f_ = os.path.join(rd, 'out', 'results.fits')
    if not os.path.exists(f_):
        _miss9.append(os.path.basename(rd)); continue
    for r in Table.read(f_):
        CFIT[str(r['id'])] = r
print(f'{len(CFIT)} fitted galaxies from {len(_rdirs) - len(_miss9)}/{len(_rdirs)} runs'
      + (f' — NOT finished: {_miss9}' if _miss9 else ''))

# prior-free baseline (the original 9b run dirs, no _Zs suffix): report how
# far the SIMBA-truth priors moved the attenuation solution
CFIT0 = {}
if _prior_runs:
    for rd in sorted(glob.glob(os.path.join(
            CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]'))):
        f_ = os.path.join(rd, 'out', 'results.fits')
        if os.path.exists(f_):
            for r in Table.read(f_):
                CFIT0[str(r['id'])] = r
    if CFIT0:
        _dav = np.array([float(CFIT[i_]['bayes.attenuation.Av_ISM'])
                         - float(CFIT0[i_]['bayes.attenuation.Av_ISM'])
                         for i_ in CFIT if i_ in CFIT0])
        print(f'[priors] Av_ISM shift vs the prior-free baseline: median '
              f'{np.median(_dav):+.3f} mag, NMAD {nmad(_dav):.3f} mag '
              f'({_dav.size} galaxies)')

_AVK = 'bayes.attenuation.generic.bessell.V'   # total V attenuation == obs V_B90
_jV9 = int(np.argmin(np.abs(GRID - 0.55)))
rows9c = []
for tid in SELECTED:
    for r in SELECTED[tid]:
        snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
        c_ = CFIT.get(f'snap{snap:03d}_gal{gid}')
        if c_ is None:
            continue
        curve = attenuation_curve(snap, gid)
        rows9c.append(dict(
            tid=tid, id=f'snap{snap:03d}_gal{gid}',
            av_rt=np.nan if curve is None else float(curve[_jV9]),
            av_ism=float(c_['bayes.attenuation.Av_ISM']),
            av_ism_err=float(c_['bayes.attenuation.Av_ISM_err']),
            av_tot=float(c_[_AVK]), av_tot_err=float(c_[f'{_AVK}_err']),
            chi2=float(c_['best.reduced_chi_square'])))
T9C = Table(rows9c)
T9C.write(os.path.join(CIGDIR, 'almac11_cigale_av_comparison.fits'),
          overwrite=True)
print(f'{len(T9C)} (target, member) pairs | median reduced chi2 = '
      f'{np.median(np.asarray(T9C["chi2"], float)):.2f}')

# ── figure 1: the Part 3 comparison, now like-for-like ──
_t9 = [t for t in TARGETS if TARGETS[t]['sample'] == 'almac11'
       and np.isfinite(TARGETS[t]['av_ism']) and t in SELECTED]
_lab9 = {'ism': 'mock CIGALE Av_ISM', 'rt': r'RT $A_V$ (Part 3)',
         'obs': 'observed CIGALE Av_ISM'}
fig, ax = plt.subplots(figsize=(10.5, 5.2))
for x, tid in enumerate(_t9):
    S = T9C[np.asarray(T9C['tid']) == tid]
    if len(S) == 0:
        continue
    y_ism = np.asarray(S['av_ism'], float)
    ax.plot(np.full(y_ism.size, x) + np.random.uniform(-0.15, 0.15, y_ism.size),
            y_ism, 'o', ms=3, color='tab:green', alpha=0.45, mec='none')
    ax.hlines(np.median(y_ism), x - 0.28, x + 0.28, color='tab:green', lw=2.2,
              label=_lab9.pop('ism', None))
    y_rt = np.asarray(S['av_rt'], float)
    y_rt = y_rt[np.isfinite(y_rt)]
    if y_rt.size:
        ax.hlines(np.median(y_rt), x - 0.28, x + 0.28, color='tab:blue',
                  lw=1.4, ls='--', label=_lab9.pop('rt', None))
    ax.errorbar(x, TARGETS[tid]['av_ism'], yerr=TARGETS[tid]['av_ism_err'],
                fmt='s', ms=6, color='k', capsize=3,
                label=_lab9.pop('obs', None))
ax.set(xticks=range(len(_t9)), xticklabels=_t9, ylabel=r'$A_V$ [mag]',
       title='consistent extraction: mock CIGALE Av_ISM (green) vs observed (black)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
ax.legend(fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_vs_cigale_consistent.png'), dpi=180)
plt.show()

# ── figure 2: estimator closure — what does CIGALE do to the RT truth? ──
m9 = np.isfinite(np.asarray(T9C['av_rt'], float))
x9 = np.asarray(T9C['av_rt'], float)[m9]
fig, ax = plt.subplots(figsize=(6.4, 5.6))
lim9 = (0.0, max(1.05 * np.nanmax(x9), 1.0))
ax.plot(lim9, lim9, color='0.6', lw=0.8)
for key, c_, lab in (('av_tot', 'tab:orange',
                      r'total (attenuation.generic.bessell.V $\equiv$ V_B90)'),
                     ('av_ism', 'tab:green', 'Av_ISM (ISM component)')):
    y9 = np.asarray(T9C[key], float)[m9]
    ax.plot(x9, y9, 'o', ms=4, color=c_, alpha=0.55, mec='none', label=lab)
    d9 = (y9 - x9)[np.isfinite(y9 - x9)]
    print(f'{lab:55s}: median offset {np.nanmedian(d9):+.2f} mag, '
          f'NMAD {nmad(d9):.2f} mag')
ax.set(xlim=lim9, ylim=(0, None),
       xlabel=r'RT truth: $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ [mag]',
       ylabel='CIGALE estimate [mag]', title='estimator closure on the mocks')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=8.5, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_cigale_closure.png'), dpi=180)
plt.show()

# ── per-target summary ──
print(f"\n{'tid':>8s} {'N':>3s} {'RT A_V':>7s} {'mock ISM':>8s} "
      f"{'mock tot':>8s} {'obs ISM':>7s}")
for tid in _t9:
    S = T9C[np.asarray(T9C['tid']) == tid]
    if len(S) == 0:
        continue
    print(f"{tid:>8s} {len(S):3d} "
          f"{np.nanmedian(np.asarray(S['av_rt'], float)):7.2f} "
          f"{np.median(np.asarray(S['av_ism'], float)):8.2f} "
          f"{np.median(np.asarray(S['av_tot'], float)):8.2f} "
          f"{TARGETS[tid]['av_ism']:7.2f}")

# ── figure 3: truth recovery — the priors' sanity check ──
# Z_star and the age floor are pinned per run; M* is NOT (it is the fitted
# normalization), so recovered-vs-caesar mass is the meaningful closure test,
# and the mass-weighted age shows what the age floor left free.
_tru = {(int(r_['SNAPSHOT']), int(r_['GROUPID_SNAPSHOT'])):
        (float(r_['LOG_MSTAR']), float(r_['AGE_STAR'])) for r_ in UNION}
_rec = []
for (snap_, gid_), (lm_, ag_) in _tru.items():
    c_ = CFIT.get(f'snap{snap_:03d}_gal{gid_}')
    if c_ is None:
        continue
    _rec.append((lm_, np.log10(max(float(c_['bayes.stellar.m_star']), 1.0)),
                 ag_, float(c_['bayes.stellar.age_m_star']) / 1e3))  # Myr->Gyr
_rec = np.array(_rec)
if _rec.size:
    fig, axs = plt.subplots(1, 2, figsize=(10.6, 4.9))
    for ax, jx, jy, lab, unit in (
            (axs[0], 0, 1, r'$\log\,M_*$', 'dex'),
            (axs[1], 2, 3, 'mass-weighted age', 'Gyr')):
        x_, y_ = _rec[:, jx], _rec[:, jy]
        lim = (min(x_.min(), np.nanmin(y_)), max(x_.max(), np.nanmax(y_)))
        ax.plot(lim, lim, color='0.6', lw=0.8)
        ax.plot(x_, y_, 'o', ms=4, color='tab:purple', alpha=0.5, mec='none')
        d_ = y_ - x_
        ax.set(xlim=lim, ylim=lim, xlabel=f'SIMBA truth {lab}',
               ylabel=f'CIGALE {lab}',
               title=f'{lab}: median {np.nanmedian(d_):+.2f} {unit}, '
                     f'NMAD {nmad(d_[np.isfinite(d_)]):.2f} {unit}')
        ax.grid(alpha=0.2, lw=0.5)
    fig.suptitle('truth recovery on the mocks (priors pin Z + age floor; '
                 'M* is fitted)', y=1.03)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, 'cigale_truth_recovery.png'), dpi=180,
                bbox_inches='tight')
    plt.show()

### Part 9d — CIGALE $A_V$ vs SIMBA $A_V$ + Part 7 with the CIGALE $A_V$

Run after 9c (`CFIT`). Two figures, both on the deduped Part-7 sample (union of
`SELECTED` members with a CIGALE fit):

- **CIGALE vs SIMBA, per galaxy** — mock CIGALE `Av_ISM` (green) and total V attenuation
  `attenuation.generic.bessell.V` (orange) against the SIMBA RT
  $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ at `findx0`; AGN hosts as triangles. The
  per-galaxy view behind 9c's closure medians: where CIGALE lands relative to the
  simulation truth, and whether AGN hosts bias differently.
- **Part 7 like-for-like** — the dust-ratio panels redone with the mock CIGALE `Av_ISM`
  on the y-axis, so the simulated points and the observed black squares are the *same
  estimator quantity* (the original Part 7 mixed the RT $A_V$ with the observed
  `Av_ISM`). Spearman printed for both `Av_ISM` and the total V attenuation.

The CIGALE fits use only `findx0` photometry, so there is no per-sightline CIGALE $A_V$ —
the sightline-resolved dust-mass figure lives in Part 8 with the RT $A_V$. Depends on
Parts 3 (`attenuation_curve`), 5 (`LOGMD_OBS`) and 9c (`CFIT`).


In [ ]:
# ── Part 9d · CIGALE A_V vs SIMBA RT A_V + Part 7 redone with CIGALE A_V ─────
# Run AFTER 9c (CFIT). Needs Parts 0 (attenuation_curve) + 5 (LOGMD_OBS).
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None
from simbanator.sed.cigale import nmad

_AVK9d = 'bayes.attenuation.generic.bessell.V'
_jV9d = int(np.argmin(np.abs(GRID - 0.55)))
_rows9d = {}                               # (snap,gid) -> selection row, deduped
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows9d.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
_recs = []
for (snap, gid), r in _rows9d.items():
    c_ = CFIT.get(f'snap{snap:03d}_gal{gid}')
    if c_ is None:
        continue
    curve = attenuation_curve(snap, gid)
    md, mg = float(r['MDUST']), float(r['MGAS'])
    _recs.append((
        np.nan if curve is None else float(curve[_jV9d]),
        float(c_['bayes.attenuation.Av_ISM']), float(c_[_AVK9d]),
        np.log10(md) - float(r['LOG_MSTAR']) if md > 0 else np.nan,
        np.log10(md / mg) if (md > 0 and mg > 0) else np.nan,
        str(r['AGN_CLASS']) == 'AGN'))
assert _recs, 'no CIGALE-fitted galaxies in SELECTED — run 9b, then 9c, first'
av_rt, av_ism, av_tot, ds9, dg9, isagn9 = map(np.asarray, zip(*_recs))
isagn9 = isagn9.astype(bool)
print(f'{len(_recs)}/{len(_rows9d)} selected galaxies have a CIGALE fit')

# ── figure 1: per-galaxy CIGALE A_V vs SIMBA RT A_V ──
m9d = np.isfinite(av_rt)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
lim = (0.0, max(1.0, 1.05 * np.nanmax(np.concatenate(
    [av_rt[m9d], av_ism, av_tot]))))
ax.plot(lim, lim, color='0.6', lw=0.8)
for y_, c_, lab in ((av_tot, 'tab:orange', 'total V (bessell.V)'),
                    (av_ism, 'tab:green', 'Av_ISM')):
    for flag, mk in ((False, 'o'), (True, '^')):
        mm = m9d & (isagn9 == flag)
        ax.plot(av_rt[mm], y_[mm], mk, ms=4.5, color=c_, alpha=0.55, mec='none',
                label=lab + (', AGN host' if flag else ''))
    d_ = (y_ - av_rt)[m9d & np.isfinite(y_)]
    print(f'{lab:22s}: median CIGALE - RT offset {np.median(d_):+.2f} mag, '
          f'NMAD {nmad(d_):.2f} mag')
ax.set(xlim=lim, ylim=(0, None),
       xlabel=r'SIMBA RT $A_V$ [mag] (findx0)', ylabel='CIGALE estimate [mag]',
       title='CIGALE $A_V$ vs SIMBA RT $A_V$ (Part 7 sample)')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=8.5, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_cigale_vs_simba.png'), dpi=180)
plt.show()

# ── figure 2: Part 7 like-for-like — dust ratios vs the CIGALE Av_ISM ──
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)
for flag, c_, lab in ((False, 'tab:blue', 'non-AGN host'),
                      (True, 'tab:orange', 'AGN host')):
    m = isagn9 == flag
    axes[0].plot(ds9[m], av_ism[m], 'o', ms=4, color=c_, alpha=0.55, mec='none',
                 label=lab)
    axes[1].plot(dg9[m], av_ism[m], 'o', ms=4, color=c_, alpha=0.55, mec='none')
for tid, T in TARGETS.items():             # observed side — as Part 7
    if T['row'] is None or not np.isfinite(T['av_ism']):
        continue
    if tid in LOGMD_OBS and np.isfinite(LOGMD_OBS[tid]):
        axes[0].errorbar(LOGMD_OBS[tid] - T['logm'], T['av_ism'], yerr=T['av_ism_err'],
                         fmt='s', ms=6, color='k', capsize=2, zorder=5)
    dgr = float(T['row'].get('logDGR', np.nan))
    if np.isfinite(dgr):
        axes[1].errorbar(dgr, T['av_ism'], yerr=T['av_ism_err'], fmt='s', ms=6,
                         color='k', capsize=2, zorder=5)
axes[1].axvline(-3, color='0.6', ls='--', lw=0.8)      # dusty-selection threshold
axes[1].axvline(-4, color='0.6', ls=':', lw=0.8)       # control threshold
axes[0].set(xlabel=r'$\log\,M_{\rm dust}/M_\star$',
            ylabel='Av_ISM [mag] — mock CIGALE (colored) / obs CIGALE (black)')
axes[1].set(xlabel=r'sim: $\log\,M_{\rm dust}/M_{\rm gas,tot}$   ·   '
                   r'obs: logDGR ($M_{\rm dust}/M_{\rm mol}$)')
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle('Part 7 like-for-like: dust ratios vs the CIGALE Av_ISM')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dust_ratio_vs_av_cigale.png'), dpi=180,
            bbox_inches='tight')
plt.show()

for yname, y_ in (('Av_ISM', av_ism), ('total V', av_tot)):
    for xname, x_ in (('log Md/M*', ds9), ('log Md/Mgas', dg9)):
        m = np.isfinite(x_) & np.isfinite(y_)
        if spearmanr is not None and m.sum() > 3:
            rho, p = spearmanr(x_[m], y_[m])
            print(f'Spearman CIGALE {yname} vs {xname}: rho={rho:+.2f} '
                  f'(p={p:.1e}, N={m.sum()})')


### Part 9e — fit quality vs dust content: is a poor $\chi^2$ a dust-poor analogue?

Theory: analogues with a low dust fraction produce no significant FIR/sub-mm emission
in powderday, so the FIR half of the mock photometry (MIPS 24, PACS, SPIRE, ALMA band 6)
gives CIGALE nothing to anchor its dust model on and the fit degrades. Plots the caesar
$M_{\rm dust}$, $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$ and
$M_{\rm dust}/M_{\rm H_2}$ against the CIGALE reduced $\chi^2$ of the mock dust_on
fits, colored by the mock ALMA band-6 flux CIGALE actually fitted (rest-frame
$\sim$0.9 mm at snaps 128–132).

In [ ]:
# ── Part 9e · fit quality vs dust content — does missing dust break the fit? ─
# Theory test: analogues with little dust produce no significant FIR/sub-mm in
# powderday, so the FIR half of the 19-band mock photometry (MIPS24, PACS,
# SPIRE, ALMA) carries no dust signal and CIGALE cannot anchor its dust model
# -> poor reduced chi2. Needs only Part 0 (reuses the 9c CFIT if that already
# ran, else loads the CIGALE results here). Dust/H2/stellar
# masses come straight from caesar; the ALMA band-6 flux is the Part 9a mock
# flux CIGALE actually fitted (observed-frame 1.3 mm -> rest ~0.9 mm at snaps
# 128-132, ~0.55 mm at the CEERS snaps 091/096).
from simbanator.io.simba import Simulation
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

CIGRUNS = os.path.join(OUTDIR, 'cigale_runs')
CIGDIR  = os.path.join(OUTDIR, 'cigale')
if 'CFIT' not in globals():             # standalone: same loader as Part 9c
    _rdirs = (sorted(glob.glob(os.path.join(CIGRUNS,
                                            'dust_on_findx0_snap*_Zs*')))
              or sorted(glob.glob(os.path.join(
                  CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]'))))
    CFIT = {}
    for rd in _rdirs:
        f_ = os.path.join(rd, 'out', 'results.fits')
        if os.path.exists(f_):
            for r in Table.read(f_):
                CFIT[str(r['id'])] = r
    print(f'[9e] CFIT loaded standalone: {len(CFIT)} galaxies from '
          f'{len(_rdirs)} run dirs')

_sim9e = Simulation('cis100')
_mass9e = {}                                 # (snap,gid) -> (Mdust, MH2, M*)
for snap in sorted({int(k[4:7]) for k in CFIT}):
    with h5py.File(_sim9e.get_caesar_file(snap), 'r') as fc_:
        d_ = fc_['galaxy_data']
        for g_, md_, mh2_, ms_ in zip(
                np.asarray(d_['GroupID'][:], int),
                np.asarray(d_['dicts/masses.dust'][:], float),
                np.asarray(d_['dicts/masses.H2'][:], float),
                np.asarray(d_['dicts/masses.stellar'][:], float)):
            _mass9e[(int(snap), int(g_))] = (float(md_), float(mh2_), float(ms_))

_f9e = Table.read(os.path.join(CIGDIR, 'almac11_fluxes_dust_on_findx0.fits'))
_alma9e = {(int(r['snap']), int(r['gal_id_at_snap'])):
           float(r['ALMA.ALMA.band6']) for r in _f9e}
_agn9e = {(int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])): bool(r['AGN_ANY'])
          for r in UNION}

_lg9e = lambda v: np.log10(v) if np.isfinite(v) and v > 0 else np.nan
rows9e = []
for key, c_ in CFIT.items():
    snap, gid = int(key[4:7]), int(key.split('_gal')[1])
    if (snap, gid) not in _mass9e:
        continue
    md, mh2, ms = _mass9e[(snap, gid)]
    rows9e.append(dict(
        snap=snap, gid=gid, chi2=float(c_['best.reduced_chi_square']),
        logmd=_lg9e(md),
        ds=_lg9e(md / ms) if ms > 0 else np.nan,
        h2s=_lg9e(mh2 / ms) if ms > 0 else np.nan,
        d2h2=_lg9e(md / mh2) if mh2 > 0 else np.nan,
        salma=_lg9e(_alma9e.get((snap, gid), np.nan)),
        agn=_agn9e.get((snap, gid), False)))
T9E = Table(rows9e)
T9E.write(os.path.join(CIGDIR, 'almac11_chi2_dustcontent.fits'), overwrite=True)
chi9e = np.asarray(T9E['chi2'], float)
sal9e = np.asarray(T9E['salma'], float)
agn9e = np.asarray(T9E['agn'], bool)
print(f'{len(T9E)} fitted galaxies | no caesar match: '
      f'{len(CFIT) - len(T9E)} | no ALMA mock flux: '
      f'{int((~np.isfinite(sal9e)).sum())} | MH2=0: '
      f'{int((~np.isfinite(np.asarray(T9E["d2h2"], float))).sum())}')

import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
fal9e = 1e3 * 10 ** sal9e                 # mock ALMA band-6 flux in micro-Jy
_fmt9e = mticker.FuncFormatter(lambda v, _p: f'{v:g}')   # plain numbers, no 10^x

# ── figure 1: the four dust/gas quantities vs reduced chi2 ──
_pan9e = (('logmd', r'$\log\,M_{\rm dust}\,[M_\odot]$'),
          ('ds',    r'$\log\,M_{\rm dust}/M_\star$'),
          ('h2s',   r'$\log\,M_{\rm H_2}/M_\star$'),
          ('d2h2',  r'$\log\,M_{\rm dust}/M_{\rm H_2}$'))
_vmin9e, _vmax9e = np.nanpercentile(fal9e, [2, 98])
_norm9e = mcolors.LogNorm(vmin=_vmin9e, vmax=_vmax9e)
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9), sharex=True)
for ax, (col, lab) in zip(axes.ravel(), _pan9e):
    y = np.asarray(T9E[col], float)
    for flag, mk, siz in ((False, 'o', 16), (True, '^', 22)):
        m = (agn9e == flag) & np.isfinite(y) & np.isfinite(chi9e) \
            & np.isfinite(sal9e)
        sc9e = ax.scatter(chi9e[m], y[m], c=fal9e[m], s=siz, marker=mk,
                          cmap='viridis', norm=_norm9e,
                          alpha=0.75, linewidths=0)
    ax.set_xscale('log')
    ax.set_ylabel(lab)
    ax.grid(alpha=0.2, lw=0.5)
    m = np.isfinite(y) & np.isfinite(chi9e)
    if spearmanr is not None and m.sum() > 3:
        rho, p = spearmanr(np.log10(chi9e[m]), y[m])
        ax.set_title(rf'Spearman $\rho={rho:+.2f}$ (p={p:.1e}, N={m.sum()})',
                     fontsize=10)
for ax in axes[1]:
    ax.set_xlabel(r'CIGALE reduced $\chi^2$ (mock dust_on, 19 bands)')
axes[0, 0].scatter([], [], marker='o', color='0.5', label='non-AGN host')
axes[0, 0].scatter([], [], marker='^', color='0.5', label='AGN host')
axes[0, 0].legend(frameon=False, fontsize=9, loc='lower left')
cb9e = fig.colorbar(sc9e, ax=axes, pad=0.02, shrink=0.92)
cb9e.set_label(r'$S_{\rm 1.3\,mm}^{\rm mock}$ [$\mu$Jy]  '
               r'(ALMA band 6, dust_on findx0)')
cb9e.ax.yaxis.set_major_formatter(_fmt9e)
cb9e.ax.yaxis.set_minor_formatter(mticker.NullFormatter())
fig.suptitle('is a poor CIGALE fit just a dust-poor analogue?', y=0.99)
fig.savefig(os.path.join(FIGDIR, 'chi2_vs_dust_content.png'), dpi=180,
            bbox_inches='tight')
plt.show()

# ── figure 2: the causal middle link — chi2 vs the sub-mm flux itself ──
ds9e = np.asarray(T9E['ds'], float)
fig, ax = plt.subplots(figsize=(6.8, 5.4))
m = np.isfinite(sal9e) & np.isfinite(chi9e) & np.isfinite(ds9e)
sc9e = ax.scatter(fal9e[m], chi9e[m], c=ds9e[m], s=16, cmap='plasma',
                  vmin=np.nanpercentile(ds9e, 2),
                  vmax=np.nanpercentile(ds9e, 98), alpha=0.75, linewidths=0)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set(xlabel=r'$S_{\rm 1.3\,mm}^{\rm mock}$ [$\mu$Jy] (ALMA band 6)',
       ylabel=r'CIGALE reduced $\chi^2$')
ax.xaxis.set_major_formatter(_fmt9e)
ax.xaxis.set_minor_formatter(mticker.NullFormatter())
ax.grid(alpha=0.2, lw=0.5)
fig.colorbar(sc9e, ax=ax, label=r'$\log\,M_{\rm dust}/M_\star$')
if spearmanr is not None and m.sum() > 3:
    rho, p = spearmanr(sal9e[m], np.log10(chi9e[m]))
    ax.set_title(rf'Spearman $\rho={rho:+.2f}$ (p={p:.1e}, N={m.sum()})',
                 fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'chi2_vs_alma_flux.png'), dpi=180)
plt.show()

# ── the theory, in one number: chi2 in the dust-poor vs dust-rich quartile ──
mq = np.isfinite(ds9e) & np.isfinite(chi9e)
q1, q3 = np.nanpercentile(ds9e[mq], [25, 75])
lo, hi = chi9e[mq & (ds9e <= q1)], chi9e[mq & (ds9e >= q3)]
print(f'log Md/M* quartiles: <= {q1:.2f} (dust-poor) | >= {q3:.2f} (dust-rich)')
print(f'median reduced chi2: dust-poor {np.median(lo):.2f} (N={lo.size}) vs '
      f'dust-rich {np.median(hi):.2f} (N={hi.size})')